# NAS - Optuna

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

## 1. Setup and Configuration

### 1.1. Environment Variables

In [1]:
import os

# Specify GPU to use (e.g., GPU:0, CPU:-1)
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Allow TensorFlow to allocate GPU memory as needed
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

# If it fails to determine best cudnn convolution algorithm
os.environ["XLA_FLAGS"] = "--xla_gpu_strict_conv_algorithm_picker=false"

In [2]:
# Disable all auto-JIT clustering at the process level
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=-1"

### 1.2. Imports

In [3]:
from _imports import * # Centralized file containing all imports

2025-08-27 14:55:21.160746: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-27 14:55:21.176789: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756317321.196382 1378108 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756317321.202370 1378108 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-08-27 14:55:21.221303: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

### 1.3. GPU Management

In [4]:
get_gpu_info()


TensorFlow GPU Monitor - 2025-08-27 14:55:23
TensorFlow Configuration
Version        : 2.18.0
CUDA Support   : Yes
CUDA Version   : 12.5.1
cuDNN Version  : 9

GPU Information
GPU Name                      Memory Usage         Temp   Util  
--------------------------------------------------------------------------------
0   NVIDIA GeForce RTX 3070      1.4GB /    8.0GB  43C    37%   



2025-08-27 14:55:23.408873: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1756317323.409836 1378108 gpu_device.cc:2022] Created device /device:GPU:0 with 4888 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070 Ti, pci bus id: 0000:b3:00.0, compute capability: 8.6


## 2. Run Parameters 

In [5]:
NUM_TRIALS = 1000
EPOCHS = 100

SAMPLER_SEED = 111

# Reproducibility settings for TensorFlow:
# Note: must have same inputs and hardware
# Warning: this affects overall performance
# tf.config.experimental.enable_op_determinism() #! Spektral does not support this yet

# Enable or disable XLA compilation
# Note: some layers don't support determinism with XLA
JIT_COMPILE = False  #! Spektral does not support this yet

In [6]:
# Number of top trials to save
TOP_K = 3

# Order to rank trials by:
# "ascending" -> the lowest value is the best
# "descending" -> the highest value is the best
ORDER = "descending"

# Key to rank trials by:
# "value" -> objective trial value
# other e.g., "test_accuracy" -> user params
RANK_KEY = "test_accuracy_s009_full"

# Direction of optimization:
# "minimize" -> the lowest value is the best
# "maximize" -> the highest value is the best
DIRECTION = "minimize"

In [7]:
POLICY = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(POLICY)

BYTES_PER_PARAM = tf.dtypes.as_dtype(POLICY.variable_dtype).size

In [8]:
# Set to an existing dir to resume training
RUN_DIR = f"runs/{get_caller_stem()}"  # (e.g. "runs/nas_1")

## 3. Data Loading and Preprocessing

In [9]:
(
    s008_coord_input,
    s008_lidar_input,
    s008_y_train,
    s009_coord_input,
    s009_lidar_input,
    s009_y,
) = load_dataset_sparse_labels()


print("s008_coord_input shape:", s008_coord_input.shape)
print("s008_lidar_input shape:", s008_lidar_input.shape)
print("s008_y_train shape:", s008_y_train.shape)
print("s009_coord_input shape:", s009_coord_input.shape)
print("s009_lidar_input shape:", s009_lidar_input.shape)
print("s009_y shape:", s009_y.shape)

/home/matheus/src/RayWise/src/_load_dataset.py:42: ComplexWarning: Casting complex values to real discards the imaginary part
  s008_y_train = s008_y_train.astype(np.float32)
/home/matheus/src/RayWise/src/_load_dataset.py:65: ComplexWarning: Casting complex values to real discards the imaginary part
  s008_y_val = s008_y_val.astype(np.float32)


Shape before conversion: (9234, 8, 32)
Shape after conversion: (9234,)
y_train shape: (9234,)
coord_input shape: (9234, 2)
lidar_input shape: (9234, 20, 200, 10)
Shape before conversion: (1960, 8, 32)
Shape after conversion: (1960,)
y_val shape: (1960,)
coord_input_val shape: (1960, 2)
lidar_input_val shape: (1960, 20, 200, 10)
y_train shape: (11194,)
coord_input shape: (11194, 2)
lidar_input shape: (11194, 20, 200, 10)
Shape before conversion: (9638, 8, 32)
Shape after conversion: (9638,)
y shape: (9638,)
coord_input shape: (9638, 2)
lidar_input shape: (9638, 20, 200, 10)
s008_coord_input shape: (11194, 2)
s008_lidar_input shape: (11194, 20, 200, 10)
s008_y_train shape: (11194,)
s009_coord_input shape: (9638, 2)
s009_lidar_input shape: (9638, 20, 200, 10)
s009_y shape: (9638,)


/home/matheus/src/RayWise/src/_load_dataset.py:101: ComplexWarning: Casting complex values to real discards the imaginary part
  s009_y = s009_y.astype(np.float32)


## 5. Model Definition

In [10]:
def build_model(trial: optuna.Trial, train_seed: int, show_summary: bool = True) -> tf.keras.Model:
    # ———————————————————————————————————————————————————————————————————————————— #
    #                              Model Construction                              #
    # ———————————————————————————————————————————————————————————————————————————— #

    # ———————————————————————————————— LiDAR Input ——————————————————————————————— #
    x_lidar_input = layers.Input(shape=(20, 200, 10), name="lidar_input")

    # Inline one-hot encoding of semantic values
    one_hot_lidar = layers.Lambda(
        lambda x: tf.concat(
            [
                # “Is there a BS anywhere in the 10 channels?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -2), axis=-1, keepdims=True), tf.float32),
                # “Vehicle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -1), axis=-1, keepdims=True), tf.float32),
                # “Obstacle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, 1), axis=-1, keepdims=True), tf.float32),
                # “Free?” → 1 channel (all channels zero)
                tf.cast(tf.reduce_all(tf.equal(x, 0), axis=-1, keepdims=True), tf.float32),
            ],
            axis=-1,
        ),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20, 200, 4),
        name="lidar_transform_to_one_hot",
    )(x_lidar_input)
    # -> (batch, 20, 200, 4)

    # Flatten the 20×200 grid into a 4000-length sequence with the 4 channels
    x_lidar_flat: layers.Layer = layers.Reshape((20 * 200, 4), name="lidar_flatten_4_channels")(one_hot_lidar)

    # ———————————————————————————————— GPS Input ———————————————————————————————— #
    # Input for coordinate data (e.g., shape: (2,))
    x_coord_input = layers.Input(shape=(2,), name="coord_input")

    # Turn (batch,2) → (batch,1,2) → tile to (batch,4000,2)'
    x_coord: layers.Layer = layers.Lambda(
        lambda x: tf.tile(tf.expand_dims(x, axis=1), [1, 20 * 200, 1]),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20 * 200, 2),
        name="coord_tile_flat",
    )(x_coord_input)

    # ————————————————————————————— Combine Branches ————————————————————————————— #
    # Fuse channels:  (batch,4000,4) + (batch,4000,2) → (batch,4000,6)
    combined = layers.Concatenate(axis=-1, name="combine_lidar_coord")([x_lidar_flat, x_coord])

    # ———————————————————————————————— Initializer ———————————————————————————————— #
    initializer = tf.keras.initializers.GlorotUniform(
        seed=train_seed,
    )

    # ———————————————————————————————————— GNN ——————————————————————————————————— #
    from spektral.layers import GraphMasking, GlobalAvgPool, ChebConv

    A = build_knn_adjacency(rows=20, cols=200, k=12)

    x_graph, a_graph = GraphMasking()([combined, A])

    gnn1 = ChebConv(
        channels=380,
        K=2,
        activation=None,
        name="cheb_1",
    )([x_graph, a_graph])
    gnn1 = layers.Activation("relu", name="cheb_act_1")(gnn1)
    gnn1 = layers.Dropout(0.25, name="cheb_drop_1")(gnn1)

    gnn2 = ChebConv(
        channels=370,
        K=2,
        activation=None,
        name="cheb_2",
    )([gnn1, a_graph])
    # This layer has no activation
    gnn2 = layers.Dropout(0.05, name="cheb_drop_2")(gnn2)

    gnn3 = ChebConv(
        channels=270,
        K=2,
        activation=None,
        name="cheb_3",
    )([gnn2, a_graph])
    gnn3 = layers.Activation("elu", name="cheb_act_3")(gnn3)
    gnn3 = layers.Dropout(0.5, name="cheb_drop_3")(gnn3)

    # skip: gnn1 -> gnn2 (concat, resize-safe)
    _gnn12 = resize_for_skip_1d(gnn1, gnn2.shape[1], name="skip_gnn1_to_gnn2_resize")
    gnn2 = layers.Concatenate(axis=-1, name="skip_from_gnn1_to_gnn2")([_gnn12, gnn2])

    # skip: gnn2 -> gnn3 (concat)
    _gnn23 = resize_for_skip_1d(gnn2, gnn3.shape[1], name="skip_gnn2_to_gnn3_resize")
    gnn3 = layers.Concatenate(axis=-1, name="skip_from_gnn2_to_gnn3")([_gnn23, gnn3])

    # ———————————————————————————————————— DNN ——————————————————————————————————— #
    # Global pooling layer to reduce the graph to a fixed-size vector
    #! The GlobalAvgPool layer from Spektral causes an error with the skip connection,
    #! it adds a singleton channel dimension to 2d tensors. Probably due to the masking.
    x = gnn3
    x = GlobalAvgPool(name="global_avg_pool")(x)

    warnings.filterwarnings("ignore", message=".*Flatten.*mask.*support masking.*")

    # Flatten to remove the singleton dimension
    x = layers.Flatten(name="flatten_gnn_output")(x)

    dnn1 = layers.Dense(units=250, activation=None, kernel_initializer=initializer, name="dense_1")(x)
    dnn1 = layers.Activation("elu", name="dense_act_1")(dnn1)

    dnn2 = layers.Dense(units=550, activation=None, kernel_initializer=initializer, name="dense_2")(dnn1)
    dnn2 = layers.Activation("tanh", name="dense_act_2")(dnn2)
    dnn2 = layers.Concatenate(axis=-1, name="skip_from_dnn1_to_dnn2")([dnn1, dnn2])

    dnn3 = layers.Dense(units=600, activation=None, kernel_initializer=initializer, name="dense_3")(dnn2)
    dnn3 = layers.Activation("tanh", name="dense_act_3")(dnn3)
    dnn3 = layers.Dropout(rate=0.2)(dnn3)
    dnn3 = layers.Concatenate(axis=-1, name="skip_from_dnn2_to_dnn3")([dnn2, dnn3])

    dnn4 = layers.Dense(units=300, activation=None, kernel_initializer=initializer, name="dense_4")(dnn3)
    dnn4 = layers.Activation("elu", name="dense_act_4")(dnn4)
    dnn4 = layers.Dropout(rate=0.4)(dnn4)

    # —————————————————————————————————— Output —————————————————————————————————— #
    outputs = layers.Dense(
        256,
        activation="softmax",
        name="output",
        kernel_initializer=initializer,
    )(x)

    # —————————————————————————— Set Inputs and Outputs —————————————————————————— #
    model = Model(inputs=(x_lidar_input, x_coord_input), outputs=(outputs,))

    # ———————————————————————————————— Compilation ——————————————————————————————— #
    model.summary() if show_summary else None

    optimizer = optimizers.Lion(learning_rate=7e-5)

    model.compile(
        optimizer=optimizer,
        loss=losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
        jit_compile=JIT_COMPILE,
    )

    return model

## 6. Objective Function

In [11]:
def objective(
    trial: optuna.Trial,
    *,
    epochs: int = 50,
    size_penalizer: Optional[str] = None,
    **kwargs: Any,
) -> float:
    """
    Objective function for Optuna to optimize a Neural Network NN on any-input data.

    Args:
        trial (optuna.Trial): Current trial for hyperparameter suggestions.
        epochs (int): Number of training epochs.
        size_penalizer (Optional[str]): type of penalizer to use:
            - "params": Penalizes based on the number of parameters.
            - "flops": Penalizes based on the number of FLOPs.
            - None: No penalization is applied.
        **kwargs: Additional keyword arguments.

    Returns:
        float: Final validation loss (optionally penalized) used for optimization.
    """
    (print(f"Running trial {trial.number}..."), clear_session())

    # ————————————————————————————— Seed Search Space ———————————————————————————— #
    DATA_SEED = trial.suggest_int("data_seed", 1, 1000, step=1)
    TRAIN_SEED = trial.suggest_int("train_seed", 1, 1000, step=1)

    # ——————————————————————————————————— Setup —————————————————————————————————— #
    # Set Python, NumPy, Keras and TensorFlow seeds
    set_random_seed(TRAIN_SEED)

    global s008_coord_input, s008_lidar_input, s008_y_train
    global s009_coord_input, s009_lidar_input, s009_y

    (
        x_s008_lidar_train,
        x_s008_lidar_val,
        x_s008_coord_train,
        x_s008_coord_val,
        y_s008_train,
        y_val,
    ) = train_test_split(
        s008_lidar_input,
        s008_coord_input,
        s008_y_train,
        test_size=0.2,
        random_state=DATA_SEED,
        shuffle=True,
    )

    (
        x_s009_lidar_test,
        x_s009_lidar_val,
        x_s009_coord_test,
        x_s009_coord_val,
        y_s009_test,
        y_s009_val,
    ) = train_test_split(
        s009_lidar_input,
        s009_coord_input,
        s009_y,
        test_size=0.2,
        random_state=DATA_SEED,
        shuffle=True,
    )

    x_lidar_train = x_s008_lidar_train
    x_coord_train = x_s008_coord_train
    y_train = y_s008_train

    x_lidar_val = np.concatenate((x_s008_lidar_val, x_s009_lidar_val), axis=0)
    x_coord_val = np.concatenate((x_s008_coord_val, x_s009_coord_val), axis=0)
    y_val = np.concatenate((y_val, y_s009_val), axis=0)

    x_lidar_test = x_s009_lidar_test
    x_coord_test = x_s009_coord_test
    y_test = y_s009_test

    backup_dir = kwargs["backup_dir"]
    model_dir = kwargs["model_dir"]
    fig_dir = kwargs["fig_dir"]
    tensorboard_dir = kwargs["tensorboard_dir"]
    logs_dir = kwargs["logs_dir"]
    history_dir = kwargs["history_dir"]

    # ———————————————————————————————————————————————————————————————————————————— #

    try:
        # ———————————————————————————————————————————————————————————————————————————— #
        #                              Data Preprocessing                              #
        # ———————————————————————————————————————————————————————————————————————————— #
        coord_scaler = StandardScaler()

        coord_scaler.fit(x_coord_train)
        x_coord_train = coord_scaler.transform(x_coord_train)
        x_coord_val = coord_scaler.transform(x_coord_val)
        x_coord_test = coord_scaler.transform(x_coord_test)
        s009_coord_input = coord_scaler.transform(s009_coord_input)

        # ———————————————————————————————————————————————————————————————————————————— #
        #                        Model Construction and Training                       #
        # ———————————————————————————————————————————————————————————————————————————— #
        model = build_model(trial=trial, train_seed=TRAIN_SEED, show_summary=True)
        BATCH_SIZE = 64

        prune_model_by_config(
            trial=trial,
            model=model,
            thresholds={
                "model_size": 80,  # Maximum model size in MB
                # "memory_mb": 8000,  # Maximum memory training usage in MB
                # "param": 1e6,  # Maximum number of parameters
                # "flops": 1e9,  # Maximum number of FLOPs
            },
            bytes_per_param=BYTES_PER_PARAM,
            batch_size=BATCH_SIZE,
        )

        history = model.fit(
            x=[x_lidar_train, x_coord_train],
            y=y_train,
            validation_data=([x_lidar_val, x_coord_val], y_val),
            epochs=epochs,
            batch_size=BATCH_SIZE,
            callbacks=get_callbacks_study(
                trial=trial,
                monitor="val_loss",
                #! Can cause high memory usage
                # tensorboard_logs=tensorboard_dir,
                reduce_lr_patience=None,
            ),
            verbose=2,
        )

        trial.set_user_attr("best_train_accuracy", float(max(history.history.get("accuracy", []))))
        trial.set_user_attr("best_val_accuracy", float(max(history.history.get("val_accuracy", []))))

        model.save(os.path.join(model_dir, f"trial_{trial.number}.keras"))

        # ———————————————————————————————————————————————————————————————————————————— #
        #                            Penalize the Model Size                           #
        # ———————————————————————————————————————————————————————————————————————————— #
        loss_values = punish_model(
            target=history.history["val_loss"],
            model=model,
            type=size_penalizer,
            flops_penalty_factor=1e-10,
            params_penalty_factor=1e-9,
            direction=DIRECTION,
        )
        loss = min(loss_values) if DIRECTION == "minimize" else max(loss_values)

        # ———————————————————————————————————————————————————————————————————————————— #
        #                              Save Trial Results                              #
        # ———————————————————————————————————————————————————————————————————————————— #

        # ——————————————————————————— Model characteristics —————————————————————————— #
        set_user_attr_model_stats(
            trial=trial,
            model=model,
            bytes_per_param=BYTES_PER_PARAM,
            batch_size=BATCH_SIZE,
            n_trials=1000,
            verbose=True,
        )

        # ————————————————————————————— Evaluate on s009 ————————————————————————————— #
        test_loss, test_acc = model.evaluate(
            [x_lidar_test, x_coord_test], y_test, batch_size=BATCH_SIZE, verbose=0
        )

        trial.set_user_attr("test_accuracy_s009", float(test_acc))
        trial.set_user_attr("test_loss_s009", float(test_loss))

        # Now evaluate on the full s009 dataset for comparison purposes
        test_loss_full, test_acc_full = model.evaluate(
            [s009_lidar_input, s009_coord_input], s009_y, batch_size=BATCH_SIZE, verbose=0
        )
        trial.set_user_attr("test_accuracy_s009_full", float(test_acc_full))
        trial.set_user_attr("test_loss_s009_full", float(test_loss_full))

        # ——————————————————————————————— Save history ——————————————————————————————— #
        history_path = os.path.join(history_dir, f"trial_{trial.number}.csv")

        # Create a DataFrame with all history data
        history_data = {
            "epoch": list(range(1, len(history.history["loss"]) + 1)),
            "train_loss": history.history["loss"],
            "val_loss": history.history["val_loss"],
        }

        # Add accuracy metrics if available
        if "accuracy" in history.history:
            history_data["train_accuracy"] = history.history["accuracy"]
        if "val_accuracy" in history.history:
            history_data["val_accuracy"] = history.history["val_accuracy"]

        # Convert to DataFrame and save as CSV
        history_df = pd.DataFrame(history_data)
        history_df.to_csv(history_path, index=False)

        # ————————————————————————— Finish the current trial ————————————————————————— #
        if len(loss_values) > 1:  # Termination Judgement Report
            report_cross_validation_scores(trial, scores=loss_values)

        return loss  # Value to minimize or maximize
    except Exception as e:
        log_trial_error(
            trial=trial,
            exc=e,
            logs_dir=logs_dir,
        )

## Main

In [ ]:
study = run_study(
    objective=objective,
    run_dir=RUN_DIR,
    epochs=EPOCHS,
    num_trials=NUM_TRIALS,
    sampler_seed=SAMPLER_SEED,
    direction=DIRECTION,
    top_k=TOP_K,
    rank_key=RANK_KEY,
    order=ORDER,
    extra_attrs=[
        "best_train_accuracy",
        "best_val_accuracy",
        "test_accuracy_s009",
        "test_accuracy_s009_full",
        "test_loss_s009",
        "test_loss_s009_full",
    ],
    variance_threshold=None,
    prune_threshold=None,
    patience=None,
)

[I 2025-08-27 14:55:25,333] A new study created in RDB with name: optuna_study


Running trial 0...


I0000 00:00:1756317325.979203 1378108 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4888 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070 Ti, pci bus id: 0000:b3:00.0, compute capability: 8.6
/home/matheus/anaconda3/envs/tf-optuna-araras/lib/python3.11/site-packages/keras/src/layers/layer.py:932: UserWarning: Layer 'skip_gnn1_to_gnn2_resize' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/home/matheus/anaconda3/envs/tf-optuna-araras/lib/python3.11/site-packages/keras/src/layers/layer.py:932: UserWarning: Layer 'skip_gnn2_to_gnn3_resize' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ lidar_input         │ (None, 20, 200,   │          0 │ -                 │
│ (InputLayer)        │ 10)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lidar_transform_to… │ (None, 20, 200,   │          0 │ lidar_input[0][0] │
│ (Lambda)            │ 4)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ coord_input         │ (None, 2)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lidar_flatten_4_ch… │ (None, 4000, 4)   │          0 │ lidar_transform_… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ coord_tile_flat     │ (None, 4000, 2)   │          0 │ coord_input[0][0] │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ combine_lidar_coord │ (None, 4000, 6)   │          0 │ lidar_flatten_4_… │
│ (Concatenate)       │                   │            │ coord_tile_flat[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ graph_masking       │ [(None, 4000, 5), │          0 │ combine_lidar_co… │
│ (GraphMasking)      │ (4000, 4000)]     │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 4000, 1)   │          0 │ combine_lidar_co… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cheb_1 (ChebConv)   │ (None, 4000, 380) │      4,180 │ graph_masking[0]… │
│                     │                   │            │ graph_masking[0]… │
│                     │                   │            │ get_item[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cheb_act_1          │ (None, 4000, 380) │          0 │ cheb_1[0][0]      │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cheb_drop_1         │ (None, 4000, 380) │          0 │ cheb_act_1[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ skip_gnn1_to_gnn2_… │ (None, 4000, 380) │          0 │ cheb_drop_1[0][0… │
│ (Lambda)            │                   │            │ get_item[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cheb_2 (ChebConv)   │ (None, 4000, 370) │    281,570 │ cheb_drop_1[0][0… │
│                     │                   │            │ graph_masking[0]… │
│                     │                   │            │ get_item[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cheb_drop_2         │ (None, 4000, 370) │          0 │ cheb_2[0][0]      │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ones_like           │ (None, 4000, 380) │          0 │ skip_gnn1_to_gnn… │
│ (OnesLike)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 4000, 381) │          0 │ ones_like[0][0],  │
│ (Concatenate)       │                   │            │ get_item[0][0]    │
├─────────────────────┼───────────────────┼────────────┼─────────────────

...

 Non-trainable params: 0 (0.00 B)

2025-08-27 14:55:26.846097: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: INVALID_ARGUMENT: Expected multiples argument to be a vector of length 4 but got length 3
/home/matheus/anaconda3/envs/tf-optuna-araras/lib/python3.11/site-packages/keras/src/layers/layer.py:932: UserWarning: Layer 'skip_gnn1_to_gnn2_resize' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/home/matheus/anaconda3/envs/tf-optuna-araras/lib/python3.11/site-packages/keras/src/models/functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ('lidar_input', 'coord_input'). Received: the structure of inputs=['*', '*']
  warnings.warn(


Instructions for updating:
This API was designed for TensorFlow v1. See https://www.tensorflow.org/guide/migrate for instructions on how to migrate your code to TensorFlow v2.


/home/matheus/anaconda3/envs/tf-optuna-araras/lib/python3.11/site-packages/keras/src/layers/layer.py:932: UserWarning: Layer 'skip_gnn2_to_gnn3_resize' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Epoch 1/100
